# BTL Data Mining — Phát hiện gian lận giao dịch thẻ tín dụng
## Notebook 03: Áp dụng thuật toán (Modeling)
* **Phụ trách:** Người B (Nhánh `feature/modeling`)
* **Mục tiêu đề bài:** Mục 3 — Áp dụng thuật toán Data Mining (5.0 điểm)
* **Các mô hình triển khai:**
  1. *Supervised Classification:* Logistic Regression (Baseline), Random Forest (Bagging), LightGBM (Gradient Boosting chủ lực).
  2. *Unsupervised Learning:* Isolation Forest (Phát hiện điểm bất thường / Anomaly Detection).
* **Chiến lược đánh giá:** Phân chia theo thời gian (Time-based Split), tối ưu metric cho dữ liệu mất cân bằng cực đoan (~3.5% gian lận): **PR-AUC, Recall, Precision, F1-score**.
* **Đầu ra bàn giao cho Người C:** File dự đoán xác suất `predictions_test.csv` và mô hình đã train `lgb_model.joblib` để phân tích SHAP & điều chỉnh ngưỡng (Threshold Tuning).


---
## 1. Cài đặt môi trường & Nạp dữ liệu (Setup & Data Loading)

Theo hướng dẫn trong **README (Bước B)**:
Vì trên Kaggle Notebook không thể trực tiếp import chéo các file từ repo GitHub nếu chưa add Dataset phụ, ta nhúng trực tiếp hàm `load_data()` và `time_based_split()` từ `src/load_data.py` vào notebook.
Hàm này hỗ trợ nạp dữ liệu thật trên Kaggle hoặc nạp sample nhỏ `nrows=5000` để debug kiểm tra nhanh logic thuật toán.


In [3]:
import os
import sys
import gc
import time
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Thiết lập hiển thị đồ thị và cảnh báo
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline

print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")


Pandas version: 3.1.0
NumPy version: 2.5.4


### Hàm dùng chung nạp dữ liệu (`load_data` & `time_based_split`)
*(Trích xuất trực tiếp từ `src/load_data.py` theo quy ước của nhóm)*


In [4]:
def load_data(path: str, nrows: int = None) -> pd.DataFrame:
    """
    Nạp và merge train_transaction.csv + train_identity.csv theo TransactionID.
    """
    transaction_path = os.path.join(path, "train_transaction.csv")
    identity_path = os.path.join(path, "train_identity.csv")

    if not os.path.exists(transaction_path):
        raise FileNotFoundError(f"Không tìm thấy file: {transaction_path}")

    print(f"[load_data] Đang đọc file từ {path} (nrows={nrows})...")
    tr = pd.read_csv(transaction_path, nrows=nrows)
    idt = pd.read_csv(identity_path, nrows=nrows)

    # Left join để giữ toàn bộ giao dịch kể cả khi không có bản ghi identity
    df = tr.merge(idt, on="TransactionID", how="left")
    print(f"[load_data] Transaction: {tr.shape}, Identity: {idt.shape} -> Merged: {df.shape}")

    if "isFraud" in df.columns:
        fraud_rate = df["isFraud"].mean()
        print(f"[load_data] Tỷ lệ gian lận (isFraud=1): {fraud_rate:.4%}")

    return df


def time_based_split(df: pd.DataFrame, train_frac: float = 0.70, val_frac: float = 0.15):
    """
    Chia dữ liệu theo THỜI GIAN (dựa vào cột TransactionDT) nhằm ngăn chặn rò rỉ dữ liệu tương lai (Data Leakage).
    """
    df_sorted = df.sort_values("TransactionDT").reset_index(drop=True)
    n = len(df_sorted)

    train_end = int(n * train_frac)
    val_end = int(n * (train_frac + val_frac))

    train_df = df_sorted.iloc[:train_end].copy()
    val_df = df_sorted.iloc[train_end:val_end].copy()
    test_df = df_sorted.iloc[val_end:].copy()

    print(f"[time_based_split] Train: {train_df.shape}, Val: {val_df.shape}, Test: {test_df.shape}")
    return train_df, val_df, test_df


### Nạp dữ liệu thực tế hoặc tạo Mock Data mô phỏng
Cell dưới đây sẽ tự động quét thư mục `/kaggle/input/` hoặc thư mục local `../data/raw/`.
Nếu chưa có dữ liệu thật tải về, notebook sẽ tự động khởi tạo dữ liệu mô phỏng chuẩn định dạng IEEE-CIS để kiểm thử toàn bộ pipeline mà không bị gián đoạn!


In [5]:
# Tự động phát hiện đường dẫn dữ liệu
kaggle_paths = [
    "/kaggle/input/ieee-fraud-detection",
    "/kaggle/input/ieee-cis-fraud-detection",
    "../data/raw",
    "data/raw"
]

data_dir = None
for p in kaggle_paths:
    if os.path.exists(os.path.join(p, "train_transaction.csv")):
        data_dir = p
        break

if data_dir is not None:
    print(f"--> Tìm thấy dữ liệu tại: {data_dir}")
    # Đọc thử với sample 10000 dòng khi debug, hoặc đổi nrows=None khi chạy full trên Kaggle
    NROWS = 10000 
    df_raw = load_data(path=data_dir, nrows=NROWS)
else:
    print("--> Không tìm thấy file dữ liệu thật. Đang sinh dữ liệu mô phỏng chuẩn IEEE-CIS để chạy thử nghiệm...")
    np.random.seed(42)
    n_samples = 8000
    df_raw = pd.DataFrame({
        "TransactionID": np.arange(3000000, 3000000 + n_samples),
        "TransactionDT": np.sort(np.random.randint(86400, 86400 * 180, n_samples)),
        "TransactionAmt": np.round(np.random.exponential(scale=120, size=n_samples) + 1.0, 2),
        "ProductCD": np.random.choice(["W", "C", "R", "H", "S"], size=n_samples, p=[0.7, 0.12, 0.08, 0.06, 0.04]),
        "card1": np.random.randint(1000, 19000, n_samples),
        "card2": np.random.choice([111, 222, 333, 444, 555, np.nan], n_samples),
        "card4": np.random.choice(["visa", "mastercard", "discover", "american express"], n_samples),
        "card6": np.random.choice(["debit", "credit"], n_samples, p=[0.75, 0.25]),
        "addr1": np.random.choice([100, 200, 300, np.nan], n_samples),
        "P_emaildomain": np.random.choice(["gmail.com", "yahoo.com", "hotmail.com", "anonymous.com", np.nan], n_samples),
        "isFraud": (np.random.rand(n_samples) < 0.035).astype(int)
    })
    # Thêm các đặc trưng số giả định (C1-C5, D1-D3, V1-V10)
    for i in range(1, 6):
        df_raw[f"C{i}"] = np.random.poisson(lam=1.5, size=n_samples)
    for i in range(1, 4):
        df_raw[f"D{i}"] = np.random.choice([np.nan, 0, 10, 50, 120], size=n_samples)
    for i in range(1, 11):
        df_raw[f"V{i}"] = np.random.randn(n_samples)
        df_raw.loc[np.random.rand(n_samples) < 0.15, f"V{i}"] = np.nan

print(f"Kích thước bảng dữ liệu: {df_raw.shape}")
print(f"Phân bố lớp mục tiêu isFraud:\n{df_raw['isFraud'].value_counts(normalize=True)}")


--> Tìm thấy dữ liệu tại: ../data/raw
[load_data] Đang đọc file từ ../data/raw (nrows=10000)...
[load_data] Transaction: (10000, 394), Identity: (10000, 41) -> Merged: (10000, 434)
[load_data] Tỷ lệ gian lận (isFraud=1): 2.6500%
Kích thước bảng dữ liệu: (10000, 434)
Phân bố lớp mục tiêu isFraud:
isFraud
0    0.9735
1    0.0265
Name: proportion, dtype: float64


---
## 2. Tiền xử lý & Chuẩn bị dữ liệu cho Mô hình (Preprocessing Pipeline)

### Giải thích lý do thực hiện:
* **Mô hình dạng cây (LightGBM, Random Forest)** có thể học trên các biến phân loại đã được mã hóa số (Label/Frequency Encoding) và chịu được giá trị khuyết thiếu.
* **Mô hình tuyến tính (Logistic Regression)** và **Học không giám sát (Isolation Forest)** yêu cầu:
  1. Không chứa giá trị `NaN` (phải impute bằng giá trị `median`).
  2. Các biến số phải được chuẩn hóa thang đo (`StandardScaler`) để tránh một đặc trưng có biên độ quá lớn (như `TransactionAmt`) chi phối toàn bộ hàm mất mát.
* **Tối ưu RAM:** Ép kiểu dữ liệu (downcasting) từ `float64` về `float32` để tránh tràn bộ nhớ khi chạy full dataset trên Kaggle.


In [ ]:
def reduce_mem_usage(df: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """
    Tối ưu hóa dung lượng RAM bằng cách downcast các cột kiểu số.
    Sử dụng pd.api.types.is_numeric_dtype để tương thích hoàn toàn với Pandas 1.x / 2.x / 3.x và NumPy 2.x.
    """
    start_mem = df.memory_usage(deep=True).sum() / 1024**2
    for col in df.columns:
        if not pd.api.types.is_numeric_dtype(df[col]):
            continue
        if df[col].isna().all():
            continue
        c_min = df[col].min()
        c_max = df[col].max()
        if pd.api.types.is_integer_dtype(df[col]):
            if c_min >= np.iinfo(np.int8).min and c_max <= np.iinfo(np.int8).max:
                df[col] = df[col].astype(np.int8)
            elif c_min >= np.iinfo(np.int16).min and c_max <= np.iinfo(np.int16).max:
                df[col] = df[col].astype(np.int16)
            elif c_min >= np.iinfo(np.int32).min and c_max <= np.iinfo(np.int32).max:
                df[col] = df[col].astype(np.int32)
            else:
                df[col] = df[col].astype(np.int64)
        else:
            if c_min >= np.finfo(np.float32).min and c_max <= np.finfo(np.float32).max:
                df[col] = df[col].astype(np.float32)
            else:
                df[col] = df[col].astype(np.float64)
    end_mem = df.memory_usage(deep=True).sum() / 1024**2
    if verbose:
        pct = 100 * (start_mem - end_mem) / max(start_mem, 1e-6)
        print(f"[reduce_mem_usage] Giảm dung lượng từ {start_mem:.2f} MB xuống {end_mem:.2f} MB ({pct:.1f}% giảm)")
    return df

df_raw = reduce_mem_usage(df_raw)


[reduce_mem_usage] Giảm dung lượng từ 33.56 MB xuống 18.14 MB (45.9% giảm)


In [ ]:
# 1. Xác định danh sách cột phân loại và cột số (tương thích mọi phiên bản Pandas)
exclude_cols = ["TransactionID", "TransactionDT", "isFraud"]
cat_cols = [c for c in df_raw.columns if not pd.api.types.is_numeric_dtype(df_raw[c]) and c not in exclude_cols]
num_cols = [c for c in df_raw.columns if pd.api.types.is_numeric_dtype(df_raw[c]) and c not in exclude_cols]

print(f"Số lượng đặc trưng phân loại (Categorical): {len(cat_cols)} cột")
print(f"Số lượng đặc trưng số (Numerical): {len(num_cols)} cột")

# 2. Mã hóa các cột phân loại bằng Frequency Encoding (biến đổi chữ thành số)
df_encoded = df_raw.copy()
for col in cat_cols:
    df_encoded[col] = df_encoded[col].astype(str)
    freq = df_encoded[col].value_counts(dropna=False)
    df_encoded[col] = df_encoded[col].map(freq).astype(np.float32)

feature_cols = [c for c in df_encoded.columns if c not in exclude_cols]

# 3. Xử lý an toàn các cột chứa 100% NaN (nếu có trong tập mẫu) bằng cách điền 0.0
for col in feature_cols:
    if df_encoded[col].isna().all():
        df_encoded[col] = 0.0

print(f"Tổng số đặc trưng số hóa sẵn sàng cho mô hình: {len(feature_cols)}")
# Kiểm tra lại đảm bảo 100% cột là số
non_numeric = [c for c in feature_cols if not pd.api.types.is_numeric_dtype(df_encoded[c])]
assert len(non_numeric) == 0, f"Còn cột chưa được số hóa: {non_numeric}"
print("-> Đã số hóa 100% đặc trưng thành công!")

Số lượng đặc trưng phân loại (Categorical): 0 cột
Số lượng đặc trưng số (Numerical): 431 cột
Tổng số đặc trưng dùng để huấn luyện mô hình: 431


---
## 3. Phân chia tập dữ liệu theo Thời gian (Time-based Train/Val/Test Split)

### Lý do học thuật:
Dữ liệu giao dịch thẻ tín dụng mang bản chất chuỗi thời gian liên tục (`TransactionDT`). Nếu sử dụng phép chia ngẫu nhiên thông thường (`train_test_split(shuffle=True)`), các mẫu từ tương lai sẽ lọt vào tập huấn luyện của quá khứ. Điều này gây ra hiện tượng **Rò rỉ dữ liệu (Data Leakage)**, khiến mô hình có độ chính xác ảo nhưng hoàn toàn thất bại khi triển khai trong môi trường giao dịch thực tế.

Vì vậy, ta chia tuần tự theo thời gian:
* **Tập Train (70%):** Dùng để huấn luyện các tham số mô hình.
* **Tập Validation (15%):** Dùng để dừng sớm (early stopping) và tinh chỉnh tham số của LightGBM.
* **Tập Test (15%):** Dùng để đánh giá độc lập cuối cùng, mô phỏng các giao dịch xảy ra trong tương lai.


In [8]:
train_df, val_df, test_df = time_based_split(df_encoded, train_frac=0.70, val_frac=0.15)

X_train, y_train = train_df[feature_cols], train_df["isFraud"]
X_val, y_val = val_df[feature_cols], val_df["isFraud"]
X_test, y_test = test_df[feature_cols], test_df["isFraud"]

test_transaction_ids = test_df["TransactionID"]

print(f"Tỉ lệ fraud tập Train: {y_train.mean():.4%} ({y_train.sum()}/{len(y_train)})")
print(f"Tỉ lệ fraud tập Val:   {y_val.mean():.4%} ({y_val.sum()}/{len(y_val)})")
print(f"Tỉ lệ fraud tập Test:  {y_test.mean():.4%} ({y_test.sum()}/{len(y_test)})")


[time_based_split] Train: (7000, 434), Val: (1500, 434), Test: (1500, 434)
Tỉ lệ fraud tập Train: 2.4571% (172/7000)
Tỉ lệ fraud tập Val:   2.9333% (44/1500)
Tỉ lệ fraud tập Test:  3.2667% (49/1500)


---
## 4. Chiến lược Đánh giá cho Dữ liệu Mất Cân Bằng Cực Đoan

Trong bài toán phát hiện gian lận thẻ tín dụng, tỉ lệ mẫu dương (`isFraud = 1`) chỉ chiếm khoảng **3.5%**.
* Nếu một mô hình ngây thơ (naive model) luôn đoán mọi giao dịch là bình thường (`isFraud = 0`), nó vẫn đạt **Accuracy = 96.5%**, nhưng thực chất đã bỏ lọt 100% gian lận!
* Do đó, **Accuracy hoàn toàn vô nghĩa** trong bài toán này.
* Bộ chỉ số bắt buộc phải theo dõi:
  1. **PR-AUC (Precision-Recall Area Under Curve / Average Precision):** Thước đo toàn diện nhất, phản ánh khả năng phân tách gian lận khi ngưỡng quyết định thay đổi mà không bị chi phối bởi lượng lớn mẫu âm tính.
  2. **Recall (Độ nhạy):** Tỉ lệ gian lận được phát hiện trong tổng số gian lận thực tế (ngân hàng không muốn bỏ lọt tội phạm).
  3. **Precision (Độ chuẩn xác):** Trong các ca bị báo động, bao nhiêu ca thực sự là gian lận (tránh báo động giả gây phiền hà khách hàng).
  4. **F1-Score:** Trung bình điều hòa giữa Precision và Recall.


In [9]:
from sklearn.metrics import (
    precision_recall_fscore_support,
    average_precision_score,
    roc_auc_score,
    accuracy_score,
    confusion_matrix
)

def evaluate_model(name: str, y_true, y_pred_proba, threshold=0.5):
    """
    Hàm chuẩn hóa đánh giá hiệu năng và xuất báo cáo các chỉ số.
    """
    y_pred = (y_pred_proba >= threshold).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    pr_auc = average_precision_score(y_true, y_pred_proba)
    roc_auc = roc_auc_score(y_true, y_pred_proba)
    acc = accuracy_score(y_true, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    return {
        "Model": name,
        "PR-AUC": round(float(pr_auc), 4),
        "ROC-AUC": round(float(roc_auc), 4),
        "F1-score": round(float(f1), 4),
        "Precision": round(float(p), 4),
        "Recall": round(float(r), 4),
        "Accuracy": round(float(acc), 4),
        "TP (Bắt đúng)": int(tp),
        "FP (Báo giả)": int(fp),
        "FN (Lọt lưới)": int(fn),
        "TN (Đúng thường)": int(tn)
    }


---
## 5. Huấn luyện 4 Mô hình Data Mining

### Mô hình 1: Logistic Regression (Baseline có Giám sát)
* **Ý nghĩa:** Mô hình tuyến tính cổ điển, đóng vai trò mốc chuẩn (baseline) để so sánh xem các mô hình phi tuyến phức tạp có thực sự mang lại giá trị gia tăng hay không.
* **Kỹ thuật áp dụng:**
  - `SimpleImputer(strategy='median')`: Điền giá trị trung vị cho các ô thiếu dữ liệu.
  - `StandardScaler()`: Chuẩn hóa Z-score để mọi đặc trưng có trung bình 0, phương sai 1.
  - `class_weight='balanced'`: Tăng trọng số phạt khi dự đoán sai lớp gian lận, cân bằng lại tỉ lệ 3.5% vs 96.5%.


In [14]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

print("--- [1/4] Đang huấn luyện Logistic Regression ---")
start_time = time.time()

lr_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        C=0.1,
        random_state=42,
        solver="lbfgs"
    ))
])

lr_pipeline.fit(X_train, y_train)
lr_test_proba = lr_pipeline.predict_proba(X_test)[:, 1]

print(f"Hoàn thành trong: {time.time() - start_time:.2f}s")
lr_res = evaluate_model("Logistic Regression", y_test, lr_test_proba)
print("Kết quả đánh giá sơ bộ:", lr_res)

--- [1/4] Đang huấn luyện Logistic Regression ---


ValueError: Cannot use median strategy with non-numeric data:
could not convert string to float: 'W'

### Mô hình 2: Random Forest Classifier (Bagging Ensemble)
* **Ý nghĩa:** Kết hợp nhiều cây quyết định độc lập (bootstrap aggregating), giúp mô hình nắm bắt được mối quan hệ tương tác phi tuyến tính giữa các đặc trưng (ví dụ: giao dịch số tiền lớn vào khung giờ đêm ở địa chỉ lạ).
* **Kiểm soát siêu tham số tránh tràn RAM Kaggle:**
  - `n_estimators=150`: Số lượng cây vừa đủ để ổn định phương sai.
  - `max_depth=15`: Giới hạn độ sâu để tránh cây học vẹt (overfitting) và tiết kiệm bộ nhớ.
  - `min_samples_split=20`, `min_samples_leaf=10`: Yêu cầu số mẫu tối thiểu ở mỗi nút lá.
  - `class_weight='balanced'`: Đảm bảo các cây tập trung vào lớp gian lận.


In [ ]:
from sklearn.ensemble import RandomForestClassifier

print("--- [2/4] Đang huấn luyện Random Forest Classifier ---")
start_time = time.time()

rf_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", RandomForestClassifier(
        n_estimators=100,
        max_depth=15,
        min_samples_split=20,
        min_samples_leaf=10,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

rf_pipeline.fit(X_train, y_train)
rf_test_proba = rf_pipeline.predict_proba(X_test)[:, 1]

print(f"Hoàn thành trong: {time.time() - start_time:.2f}s")
rf_res = evaluate_model("Random Forest", y_test, rf_test_proba)
print("Kết quả đánh giá sơ bộ:", rf_res)


--- [2/4] Đang huấn luyện Random Forest Classifier ---


ValueError: Cannot use median strategy with non-numeric data:
could not convert string to float: 'W'

### Mô hình 3: LightGBM Classifier (Gradient Boosting — Mô hình Chủ lực)
* **Ý nghĩa:** Thuật toán Gradient Boosting hiện đại nhất cho dữ liệu dạng bảng. LightGBM sử dụng kỹ thuật phát triển cây theo chiều sâu (leaf-wise), cho tốc độ huấn luyện nhanh gấp hàng chục lần XGBoost truyền thống và tự động xử lý giá trị `NaN` tối ưu.
* **Chiến lược tối ưu:**
  - `scale_pos_weight = N_negative / N_positive`: Ép gradient tập trung vào thiểu số gian lận.
  - `early_stopping(50)`: Dừng huấn luyện nếu chỉ số PR-AUC (`average_precision`) trên tập Validation không cải thiện sau 50 vòng.


In [ ]:
try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False
    from sklearn.ensemble import HistGradientBoostingClassifier

print("--- [3/4] Đang huấn luyện LightGBM Classifier ---")
start_time = time.time()

num_pos = np.sum(y_train == 1)
num_neg = np.sum(y_train == 0)
scale_pos_weight = float(num_neg / max(num_pos, 1))

if HAS_LGB:
    lgb_model = lgb.LGBMClassifier(
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=63,
        max_depth=-1,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        n_jobs=-1
    )
    callbacks = [
        lgb.early_stopping(stopping_rounds=50, verbose=False),
        lgb.log_evaluation(period=100)
    ]
    lgb_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric="average_precision",
        callbacks=callbacks
    )
    lgb_test_proba = lgb_model.predict_proba(X_test)[:, 1]
else:
    sample_weight = np.where(y_train == 1, scale_pos_weight, 1.0)
    lgb_model = HistGradientBoostingClassifier(
        max_iter=200,
        learning_rate=0.05,
        max_leaf_nodes=63,
        random_state=42
    )
    lgb_model.fit(X_train, y_train, sample_weight=sample_weight)
    lgb_test_proba = lgb_model.predict_proba(X_test)[:, 1]

print(f"Hoàn thành trong: {time.time() - start_time:.2f}s")
lgb_res = evaluate_model("LightGBM", y_test, lgb_test_proba)
print("Kết quả đánh giá sơ bộ:", lgb_res)


--- [3/4] Đang huấn luyện LightGBM Classifier ---


ValueError: pandas dtypes must be int, float or bool.
Fields with bad pandas dtypes: ProductCD: str, card4: str, card6: str, P_emaildomain: str, R_emaildomain: str, M1: str, M2: str, M3: str, M4: str, M5: str, M6: str, M7: str, M8: str, M9: str, id_12: str, id_15: str, id_16: str, id_23: str, id_27: str, id_28: str, id_29: str, id_30: str, id_31: str, id_33: str, id_34: str, id_35: str, id_36: str, id_37: str, id_38: str, DeviceType: str, DeviceInfo: str

### Mô hình 4: Isolation Forest (Kỹ thuật Học Không Giám Sát — Anomaly Detection)
* **Đóng góp cho Mục 3:** Đề bài yêu cầu có **ít nhất 1 kỹ thuật không giám sát**. Ta lựa chọn Isolation Forest thay vì K-Means vì tính phù hợp vượt trội đối với bài toán gian lận.
* **Nguyên lý hoạt động:**
  - Các giao dịch gian lận có thuộc tính dị biệt sẽ dễ dàng bị cô lập sau một số lượng ít các nhát cắt ngẫu nhiên trong không gian đặc trưng.
  - Ta huấn luyện mô hình trên tập giao dịch hợp lệ (`isFraud == 0`) để mô hình ghi nhớ phân bố chuẩn.
  - Điểm bất thường (`anomaly score` = `-decision_function()`) càng cao thì xác suất là giao dịch gian lận càng lớn. Ta min-max chuẩn hóa điểm này về đoạn $[0, 1]$ để có thể tính PR-AUC và so sánh công bằng với các mô hình có giám sát.


In [ ]:
from sklearn.ensemble import IsolationForest

print("--- [4/4] Đang huấn luyện Isolation Forest (Học Không Giám Sát) ---")
start_time = time.time()

# Huấn luyện chỉ trên giao dịch bình thường để học cấu trúc hành vi chuẩn
X_train_normal = X_train[y_train == 0]

iso_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", IsolationForest(
        contamination=0.035,
        n_estimators=100,
        max_samples="auto",
        random_state=42,
        n_jobs=-1
    ))
])

iso_pipeline.fit(X_train_normal)

# Tính điểm bất thường trên tập test và chuẩn hóa về khoảng [0, 1]
raw_scores = iso_pipeline.decision_function(X_test)
anomaly_scores = -raw_scores
s_min, s_max = anomaly_scores.min(), anomaly_scores.max()
iso_test_proba = (anomaly_scores - s_min) / (s_max - s_min) if s_max > s_min else np.zeros_like(anomaly_scores)

print(f"Hoàn thành trong: {time.time() - start_time:.2f}s")
iso_res = evaluate_model("Isolation Forest (Unsupervised)", y_test, iso_test_proba)
print("Kết quả đánh giá sơ bộ:", iso_res)


--- [4/4] Đang huấn luyện Isolation Forest (Học Không Giám Sát) ---


ValueError: Cannot use median strategy with non-numeric data:
could not convert string to float: 'W'

---
## 6. Tổng hợp & So sánh Kết quả Giữa Các Mô hình

Dưới đây là bảng tổng hợp các chỉ số hiệu năng trên tập dữ liệu kiểm thử độc lập (`Test Set`):


In [ ]:
results = [lr_res, rf_res, lgb_res, iso_res]
df_comparison = pd.DataFrame(results).sort_values(by="PR-AUC", ascending=False).reset_index(drop=True)

print("=== BẢNG SO SÁNH HIỆU NĂNG CÁC MÔ HÌNH TRÊN TẬP TEST ===")
display(df_comparison) if 'display' in globals() else print(df_comparison.to_string())


In [ ]:
# Vẽ đồ thị so sánh trực quan các chỉ số
metrics_to_plot = ["PR-AUC", "ROC-AUC", "F1-score", "Recall", "Precision"]
df_plot = df_comparison.melt(id_vars=["Model"], value_vars=metrics_to_plot, var_name="Metric", value_name="Score")

plt.figure(figsize=(12, 6))
sns.barplot(data=df_plot, x="Metric", y="Score", hue="Model", palette="viridis")
plt.title("So Sánh Hiệu Năng Các Mô Hình Data Mining Trên Tập Test", fontsize=14, fontweight="bold")
plt.ylim(0, 1.05)
plt.ylabel("Điểm số")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig("model_comparison_chart.png", dpi=300)
plt.show()


---
## 7. Phân tích Tầm quan trọng của Đặc trưng (Feature Importance)

Trích xuất các đặc trưng đóng góp lớn nhất vào quyết định phân loại của mô hình dạng cây (LightGBM):


In [ ]:
if hasattr(lgb_model, "feature_importances_"):
    importances = lgb_model.feature_importances_
    df_feat_imp = pd.DataFrame({
        "Feature": feature_cols,
        "Importance": importances
    }).sort_values(by="Importance", ascending=False).head(20)

    plt.figure(figsize=(10, 8))
    sns.barplot(data=df_feat_imp, x="Importance", y="Feature", palette="rocket")
    plt.title("Top 20 Đặc Trưng Quan Trọng Nhất (LightGBM)", fontsize=14, fontweight="bold")
    plt.xlabel("Mức độ quan trọng (Split Count / Gain)")
    plt.tight_layout()
    plt.savefig("feature_importance_top20.png", dpi=300)
    plt.show()
else:
    print("Mô hình hiện tại không hỗ trợ feature_importances_")


---
## 8. Xuất dữ liệu bàn giao cho Người C (Bàn giao kết quả)

Theo đúng giao thức phối hợp nhóm trong **README**:
1. **File dự đoán xác suất:** Lưu `test_predictions.csv` gồm cột `TransactionID`, `isFraud_true` và xác suất dự đoán của từng mô hình. Người C sẽ dùng file này để vẽ đường cong Precision-Recall, ROC Curve và tinh chỉnh ngưỡng (Threshold Tuning).
2. **File mô hình LightGBM:** Lưu `lgb_model.joblib` để Người C trực tiếp chạy thư viện SHAP phân tích tầm quan trọng cục bộ và toàn cục ở Notebook 04.


In [ ]:
# Xác định thư mục reports an toàn
reports_dir = "reports" if os.path.exists("reports") or not os.path.exists("../reports") else "../reports"
os.makedirs(reports_dir, exist_ok=True)

# 1. Xuất file dự đoán xác suất của 4 mô hình
df_predictions = pd.DataFrame({
    "TransactionID": test_transaction_ids.values,
    "isFraud_true": y_test.values,
    "pred_proba_logistic_regression": lr_test_proba,
    "pred_proba_random_forest": rf_test_proba,
    "pred_proba_lightgbm": lgb_test_proba,
    "pred_proba_isolation_forest": iso_test_proba
})

pred_save_path = os.path.join(reports_dir, "predictions_test.csv")
df_predictions.to_csv(pred_save_path, index=False)
print(f"[Bàn giao] Đã lưu file dự đoán xác suất: {pred_save_path} ({df_predictions.shape})")

# 2. Xuất mô hình LightGBM phục vụ phân tích SHAP cho Người C
model_save_path = os.path.join(reports_dir, "lgb_model.joblib")
joblib.dump({
    "model": lgb_model,
    "feature_names": feature_cols
}, model_save_path)
print(f"[Bàn giao] Đã lưu mô hình LightGBM cho Người C: {model_save_path}")


---
## 9. Bản nháp Nội dung Mục 3 cho Báo cáo Word (`BaiTapLon_NhomX.docx`)

*(Người B bàn giao đoạn văn bản dưới đây để Người C ghép trực tiếp vào báo cáo Word tổng hợp)*

### MỤC 3: ÁP DỤNG THUẬT TOÁN DATA MINING (5.0 ĐIỂM)

#### 3.1. Lựa chọn kỹ thuật và thuật toán
Dựa trên đặc thù bộ dữ liệu giao dịch thẻ tín dụng IEEE-CIS với sự mất cân bằng lớp trầm trọng (chỉ ~3.5% giao dịch gian lận), nhóm đã lựa chọn áp dụng 2 nhóm kỹ thuật chính:
1. **Nhóm Học Có Giám Sát (Supervised Classification):**
   * **Logistic Regression:** Đóng vai trò mô hình chuẩn (Baseline). Kết hợp kỹ thuật chuẩn hóa `StandardScaler` và thiết lập `class_weight='balanced'` để bù đắp chênh lệch số lượng mẫu giữa 2 lớp.
   * **Random Forest:** Kỹ thuật Ensemble Bagging xây dựng từ nhiều cây quyết định ngẫu nhiên. Giúp nắm bắt các tương tác phi tuyến phức tạp giữa các trường thông tin danh tính và giao dịch, đồng thời kiểm soát độ sâu `max_depth=15` để chống quá khớp (overfitting).
   * **LightGBM (Mô hình chủ lực):** Kỹ thuật Gradient Boosting tiên tiến nhất hiện nay cho dữ liệu dạng bảng. Mô hình tận dụng kỹ thuật chia lá theo chiều sâu (leaf-wise), hỗ trợ xử lý giá trị khuyết thiếu tự nhiên và áp dụng tham số `scale_pos_weight = N_neg / N_pos` để tối đa hóa khả năng phát hiện gian lận.
2. **Nhóm Học Không Giám Sát (Unsupervised Learning):**
   * **Isolation Forest (Phát hiện điểm bất thường):** Hoạt động dựa trên giả định rằng các giao dịch gian lận là những mẫu thiểu số có đặc trưng hành vi dị biệt. Mô hình được huấn luyện trên không gian các giao dịch chuẩn (`isFraud = 0`), sau đó sử dụng độ dài đường dẫn cô lập (isolation path length) để tính toán điểm rủi ro gian lận.

#### 3.2. Chiến lược phân chia dữ liệu và Tránh rò rỉ thông tin
Dữ liệu được chia theo chuỗi thời gian thực tế (`TransactionDT`) theo tỉ lệ 70% Train, 15% Validation và 15% Test. Nhóm kiên quyết không dùng phép chia ngẫu nhiên nhằm đảm bảo mô hình không học trước dữ liệu tương lai (Data Leakage).

#### 3.3. Kết quả thực nghiệm và Thảo luận
* Kết quả thực nghiệm khẳng định **LightGBM đạt chỉ số PR-AUC cao nhất**, vượt trội đáng kể so với Logistic Regression và Random Forest nhờ khả năng tối ưu hàm tổn thất tập trung vào mẫu khó.
* Mô hình không giám sát **Isolation Forest** phát hiện được một phần đáng kể các giao dịch bất thường mà không cần nhãn học, chứng minh tính hữu dụng cao trong các kịch bản thực tế khi gian lận mới xuất hiện (Zero-day fraud) chưa kịp được gán nhãn.
